# Libs

In [1]:
import os

import numpy as np
import pandas as pd

# Load

In [2]:
if os.path.exists("../data/raw/online_retail_II.parquet"):
    df_retail = pd.read_parquet("../data/raw/online_retail_II.parquet")
else:
    df_2009_2010 = pd.read_excel(
        "../data/raw/online_retail_II.xlsx",
        sheet_name="Year 2009-2010"
    )

    df_2010_2011 = pd.read_excel(
        "../data/raw/online_retail_II.xlsx",
        sheet_name="Year 2010-2011"
    )

    df_retail = pd.concat(
        [df_2009_2010, df_2010_2011],
        ignore_index=True
    )

    df_retail["Invoice"] = df_retail["Invoice"].astype(str)
    df_retail["StockCode"] = df_retail["StockCode"].astype(str)
    df_retail["Description"] = df_retail["Description"].astype(str)
    
    df_retail.to_parquet("../data/raw/online_retail_II.parquet", index=False)

In [3]:
rename_col = {
    "Invoice": "invoice_id",
    "StockCode": "product_id",
    "Description": "product_description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_time",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
}
df_retail = df_retail.rename(columns=rename_col)
df_retail["total_price"] = df_retail["quantity"] * df_retail["unit_price"]
df_retail["invoice_date"] = df_retail["invoice_time"].dt.date
df_retail["invoice_month"] = df_retail["invoice_time"].dt.to_period("M").dt.to_timestamp()

# Data Quality Checks

In [4]:
df_retail.head()

,invoice_id,product_id,product_description,quantity,invoice_time,unit_price,customer_id,country,total_price,invoice_date,invoice_month
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12-01,2009-12-01
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01,2009-12-01
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01,2009-12-01
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12-01,2009-12-01
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12-01,2009-12-01


In [5]:
df_retail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 11 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   invoice_id           1067371 non-null  object        
 1   product_id           1067371 non-null  object        
 2   product_description  1067371 non-null  object        
 3   quantity             1067371 non-null  int64         
 4   invoice_time         1067371 non-null  datetime64[ns]
 5   unit_price           1067371 non-null  float64       
 6   customer_id          824364 non-null   float64       
 7   country              1067371 non-null  object        
 8   total_price          1067371 non-null  float64       
 9   invoice_date         1067371 non-null  object        
 10  invoice_month        1067371 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(3), int64(1), object(5)
memory usage: 89.6+ MB


In [6]:
df_retail["invoice_time"].min(), df_retail["invoice_time"].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:50:00'))

In [7]:
df_retail["invoice_id"].isnull().sum(), df_retail["invoice_id"].nunique()

(np.int64(0), 53628)

In [8]:
df_retail["product_id"].isnull().sum(), df_retail["product_id"].nunique()

(np.int64(0), 5305)

# Customer Repurchase Analysis

In [9]:
df_cust_repurchase = df_retail.groupby("customer_id")["invoice_id"] \
    .nunique().reset_index()
df_retail["customer_id"].isnull().sum(), \
    df_retail["customer_id"].nunique(), \
    df_cust_repurchase[df_cust_repurchase["invoice_id"] > 1].shape[0], \
    100 * df_cust_repurchase[df_cust_repurchase["invoice_id"] > 1].shape[0]  \
        / df_retail["customer_id"].nunique()

(np.int64(243007), 5942, 4481, 75.41231908448334)

In [10]:
df_cust_repurchase[["invoice_id"]].describe(
    percentiles=[0.01, 0.05] + [i/10 for i in range(1, 10)] + [0.95, 0.99]
).round(2)

,invoice_id
count,5942.00
mean,7.55
std,15.97
min,1.00
1%,1.00
5%,1.00
10%,1.00
20%,1.00
30%,2.00
40%,3.00


# Feature Engineering

In [11]:
df_retail["invoice_time"] = pd.to_datetime(df_retail["invoice_time"])

# Invoice-level table: one row per invoice
df_inv = (
    df_retail.dropna(subset=["customer_id"])
    .groupby(["customer_id", "invoice_id"], as_index=False)["invoice_time"].min()
    .sort_values(["customer_id", "invoice_time", "invoice_id"])
)

# Order sequence per customer (1 = first purchase)
df_inv["order_seq"] = df_inv.groupby("customer_id").cumcount() + 1
df_inv["customer_type"] = np.where(df_inv["order_seq"] == 1, "new", "returning")

# Merge back into the transaction table
df_retail = df_retail.merge(
    df_inv[["customer_id", "invoice_id", "order_seq", "customer_type"]],
    on=["customer_id", "invoice_id"],
    how="left",
)

# Rows without a Customer ID -> cannot be determined
df_retail["customer_type"] = df_retail["customer_type"].fillna("unknown")

In [19]:
df_cust_monthly = df_retail.groupby(["invoice_month", "customer_type"])["customer_id"].nunique().reset_index().pivot(
    index="invoice_month", columns="customer_type", values="customer_id"
).reset_index().rename_axis(None, axis=1).fillna(0)

In [21]:
df_cust_monthly["invoice_month"] = df_cust_monthly["invoice_month"].dt.strftime("%Y-%m")

In [28]:
df_cust_monthly.drop(columns="unknown").set_index("invoice_month").T.style \
    .background_gradient(cmap="Blues", axis=1).format("{:,.0f}")

invoice_month,2009-12,2010-01,2010-02,2010-03,2010-04,2010-05,2010-06,2010-07,2010-08,2010-09,2010-10,2010-11,2010-12,2011-01,2011-02,2011-03,2011-04,2011-05,2011-06,2011-07,2011-08,2011-09,2011-10,2011-11,2011-12
new,"1,045",394,363,436,291,254,269,183,158,242,379,322,77,71,123,178,105,108,108,102,106,187,221,192,28
returning,413,476,525,782,758,859,887,852,845,"1,012","1,273","1,452",883,726,695,879,810,998,966,911,896,"1,155","1,251","1,565",660


In [31]:
df_cust_monthly.drop(columns="unknown").set_index("invoice_month").T \
    .div(df_cust_monthly.set_index("invoice_month").T.sum(axis=0), axis=1) \
    .style.background_gradient(cmap="Blues", axis=1).format("{:.1%}")

invoice_month,2009-12,2010-01,2010-02,2010-03,2010-04,2010-05,2010-06,2010-07,2010-08,2010-09,2010-10,2010-11,2010-12,2011-01,2011-02,2011-03,2011-04,2011-05,2011-06,2011-07,2011-08,2011-09,2011-10,2011-11,2011-12
new,71.7%,45.3%,40.9%,35.8%,27.7%,22.8%,23.3%,17.7%,15.8%,19.3%,22.9%,18.2%,8.0%,8.9%,15.0%,16.8%,11.5%,9.8%,10.1%,10.1%,10.6%,13.9%,15.0%,10.9%,4.1%
returning,28.3%,54.7%,59.1%,64.2%,72.3%,77.2%,76.7%,82.3%,84.2%,80.7%,77.1%,81.8%,92.0%,91.1%,85.0%,83.2%,88.5%,90.2%,89.9%,89.9%,89.4%,86.1%,85.0%,89.1%,95.9%


In [37]:
df_trans_monthly = df_retail[["invoice_month", "customer_type"]].value_counts().reset_index()

In [39]:
df_trans_monthly["invoice_month"] = df_trans_monthly["invoice_month"].dt.strftime("%Y-%m")

In [44]:
df_trans_monthly.pivot(index="invoice_month", columns="customer_type", values="count").reset_index().rename_axis(None, axis=1).fillna(0) \
    .set_index("invoice_month").T.style.background_gradient(cmap="Blues", axis=1).format("{:,.0f}")

invoice_month,2009-12,2010-01,2010-02,2010-03,2010-04,2010-05,2010-06,2010-07,2010-08,2010-09,2010-10,2010-11,2010-12,2011-01,2011-02,2011-03,2011-04,2011-05,2011-06,2011-07,2011-08,2011-09,2011-10,2011-11,2011-12
new,"20,143","8,546","8,416","10,373","6,273","5,849","7,155","4,478","4,098","5,762","9,988","9,728","3,149","1,914","2,567","4,134","2,361","2,158","2,685","2,838","2,653","4,652","6,746","5,408",640
returning,"11,617","13,893","15,490","22,741","21,560","23,755","24,795","23,268","22,844","29,624","40,573","51,762","38,504","19,998","17,796","23,688","20,837","26,750","25,151","24,664","25,009","36,170","43,949","60,190","17,022"
unknown,"13,468","9,116","5,482","8,397","6,224","5,719","8,033","5,637","6,364","6,705","8,537","16,525","23,351","13,235","7,344","8,926","6,718","8,122","9,038","12,016","7,622","9,404","10,047","19,113","7,864"
